# LLM Uncertainty Benchmark — Kaggle GPU

Runs the full CP benchmark with **open-source models on Kaggle's free GPU** (T4 x2 or P100).  
Uses HuggingFace Transformers for **direct logit extraction** — more accurate than continuation scoring.

**Models:** Qwen3-0.6B, Qwen2.5-3B, Gemma-3-1B, Llama-3.2-1B  
*(all fit in 16GB GPU RAM; larger models need GPU T4 x2)*

**Setup:**
1. New notebook → Settings → Accelerator → **GPU T4 x2**
2. Add secret: `HF_TOKEN` (from huggingface.co → Settings → Access Tokens)
3. Run all cells

**Time estimate:** ~1–2 hours for 4 models × 5 tasks × n=100

## 1. Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'bitsandbytes',
                'numpy', 'scikit-learn', 'tqdm', 'matplotlib'], check=True)
print('Done.')

## 2. Clone repo

In [ ]:
import os, subprocess

if not os.path.exists('LLM-Uncertainty-Study'):
    subprocess.run(['git', 'clone', 'https://github.com/SokhengDin/LLM-Uncertainty-Study.git'], check=True)
os.chdir('LLM-Uncertainty-Study')
print('Working directory:', os.getcwd())

## 3. Config — models and tasks

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# HuggingFace token (needed for gated models like Llama)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print('HF_TOKEN not found in secrets — gated models may fail')

# ── Models — all fit on T4 16GB ──────────────────────────────────────────────
# Format: (hf_model_id, short_name)
# All models below are fully PUBLIC (no HF gating, no approval needed):
#   - Qwen3-0.6B, Qwen2.5-3B  → Apache 2.0, ungated
#   - gemma-2b-it              → Google license, ungated (gemma-3-* are gated)
#   - OLMo-1B-Instruct         → Apache 2.0, fully open, replaces gated Llama
MODELS = [
    ('Qwen/Qwen3-0.6B',                    'qwen3-0.6b'),
    ('Qwen/Qwen2.5-3B-Instruct',           'qwen2.5-3b'),
    ('google/gemma-2b-it',                 'gemma-2b'),
    ('allenai/OLMo-1B-hf',                 'olmo-1b'),
    # Uncomment for T4 x2 (32GB total) — all ungated:
    # ('Qwen/Qwen2.5-7B-Instruct',         'qwen2.5-7b'),
    # ('google/gemma-7b-it',               'gemma-7b'),
    # ('allenai/OLMo-7B-Instruct-hf',      'olmo-7b'),
    # ── Gated (need HF approval + HF_TOKEN secret) ──
    # ('meta-llama/Llama-3.2-1B-Instruct', 'llama3.2-1b'),
    # ('meta-llama/Llama-3.1-8B-Instruct', 'llama3.1-8b'),
    # ('google/gemma-3-1b-it',             'gemma-3-1b'),
]

DATASETS = [
    'mmlu_10k',
    'cosmosqa_10k',
    'hellaswag_10k',
    'halu_dialogue',
    'halu_summarization',
]

SAMPLES  = 100    # n_cal ≈ 52 → qhat ≈ 0.923 (non-trivial CP)
ALPHA    = 0.1
PROMPT   = 'base'
ICL      = 'icl1'
OUT_DIR  = 'outputs_kaggle'
FIG_DIR  = 'figures_kaggle'

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Config: {len(MODELS)} models × {len(DATASETS)} tasks × n={SAMPLES}')
print('Models:', [m[1] for m in MODELS])

## 4. HuggingFace logit extractor
Direct last-token logits — more accurate than continuation scoring.

In [ ]:
import json, pickle, random, sys
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

CHOICES = ['A', 'B', 'C', 'D', 'E', 'F']

FEW_SHOT_IDS = {
    'MMLU'               : [1, 3, 5, 7, 9],
    'HellaSwag'          : [1, 3, 5, 7, 9],
    'CosmosQA'           : [1, 3, 5, 7, 9],
    'Halu-OpenDialKG'    : [5, 7, 9],
    'Halu-CNN/DailyMail' : [9],
}
FEW_SHOT_RESERVE = 10
IDS_TO_REMOVE    = [1, 3, 5, 7, 9]  # same as main.py


def load_data(path, max_samples):
    data = json.load(open(path))
    few  = data[:FEW_SHOT_RESERVE]
    rest = data[FEW_SHOT_RESERVE:]
    if len(rest) > max_samples:
        random.seed(42)
        rest = random.sample(rest, max_samples)
    print(f'  {FEW_SHOT_RESERVE} few-shot + {len(rest)} test = {len(few+rest)} total')
    return few + rest


def fmt_example(ex, prompt, with_answer=False):
    src = ex['source']
    if src == 'MMLU':
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src in ('CosmosQA', 'HellaSwag'):
        prompt += 'Context: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src == 'Halu-OpenDialKG':
        prompt += 'Dialogue: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src == 'Halu-CNN/DailyMail':
        prompt += 'Document: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    for k, v in ex['choices'].items():
        prompt += f'{k}. {v}\n'
    prompt += 'Answer:'
    if with_answer:
        prompt += ' ' + ex['answer'] + '\n'
    return prompt


def build_prompts(data):
    src   = data[0]['source']
    fsids = FEW_SHOT_IDS[src]
    shots = [data[i] for i in fsids]
    out   = []
    for ex in data:
        if ex['id'] in IDS_TO_REMOVE:
            continue
        p = ''
        for fs in shots:
            p = fmt_example(fs, p, with_answer=True)
        out.append({'id': ex['id'], 'prompt': fmt_example(ex, p)})
    return out


class HFLogitExtractor:
    """Extract last-token logprobs for A-F using HuggingFace model."""

    def __init__(self, model_id, hf_token=''):
        print(f'Loading {model_id}...')
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, token=hf_token or None, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, token=hf_token or None,
            torch_dtype=torch.float16,
            device_map='auto',
            trust_remote_code=True,
        )
        self.model.eval()

        # Get token IDs for A-F (take last token of each option string)
        self.choice_ids = [
            self.tokenizer.encode(f' {c}', add_special_tokens=False)[-1]
            for c in CHOICES
        ]
        print(f'  Choice token IDs: {dict(zip(CHOICES, self.choice_ids))}')

    def get_choice_logits(self, prompt: str) -> np.ndarray:
        inputs = self.tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=2048
        ).to(self.model.device)
        with torch.no_grad():
            out = self.model(**inputs)
        # Last token logits over full vocab
        last_logits = out.logits[0, -1, :].float().cpu()
        return last_logits[self.choice_ids].numpy()

    def unload(self):
        import gc
        del self.model
        gc.collect()
        torch.cuda.empty_cache()
        print('  Model unloaded from GPU.')


print('Logit extractor ready.')

## 5. Generate logits — all models × all tasks

In [ ]:
for hf_id, short in MODELS:
    # Check if all datasets already done
    all_done = all(
        os.path.exists(f'{OUT_DIR}/{short}_{ds}_base_icl1_sample{SAMPLES}.pkl')
        for ds in DATASETS
    )
    if all_done:
        print(f'SKIP {short} (all datasets done)')
        continue

    extractor = HFLogitExtractor(hf_id, HF_TOKEN)

    for ds in DATASETS:
        pkl = f'{OUT_DIR}/{short}_{ds}_base_icl1_sample{SAMPLES}.pkl'
        if os.path.exists(pkl):
            print(f'  SKIP {short} | {ds}')
            continue

        print(f'\n--- {short} | {ds} ---')
        data    = load_data(f'data/{ds}.json', SAMPLES)
        prompts = build_prompts(data)

        outputs = []
        for ex in tqdm(prompts, desc=f'{short}|{ds}'):
            logits = extractor.get_choice_logits(ex['prompt'])
            outputs.append({'id': ex['id'], 'logits_options': logits})

        with open(pkl, 'wb') as f:
            pickle.dump(outputs, f)
        print(f'  Saved → {pkl}')

    extractor.unload()

print('\nAll logits generated!')

## 6. CP evaluation

In [ ]:
import subprocess, sys

for hf_id, short in MODELS:
    print(f'\n=== Evaluating {short} ===')
    subprocess.run([
        sys.executable, 'main.py',
        '--model',           short,
        '--data_names',      *DATASETS,
        '--prompt_methods',  PROMPT,
        '--icl_methods',     ICL,
        '--max_samples',     str(SAMPLES),
        '--alpha',           str(ALPHA),
        '--logits_data_dir', OUT_DIR,
        '--output_dir',      OUT_DIR,
    ], check=False)

## 6b. Mondrian CP — entropy-stratified thresholds (D1 research direction)

Instead of one global q̂ for all questions, compute a separate threshold per
difficulty bin defined by entropy quantiles. This gives **conditional coverage
per bin** (≥90% in each bin) rather than only marginal coverage on average.

Comparison:
- **Global CP** (Ye et al.) → one q̂, CR satisfied on average
- **Mondrian CP** (this cell) → one q̂ per entropy bin, CR satisfied within each bin

In [ ]:
import json, pickle, random
import numpy as np
import sys
sys.path.insert(0, '.')

from utils.mondrian_cp import evaluate_mondrian

OPTIONS = ['A', 'B', 'C', 'D', 'E', 'F']

def load_split(short, ds, out_dir, samples, cal_frac=0.5, seed=42):
    """Load logits pkl and split into cal / test matching main.py logic."""
    pkl = f'{out_dir}/{short}_{ds}_base_icl1_sample{samples}.pkl'
    if not os.path.exists(pkl):
        return None, None, None, None

    logits_all = pickle.load(open(pkl, 'rb'))

    data_all = json.load(open(f'data/{ds}.json'))
    # replicate main.py sampling: reserve 10 few-shot, sample rest
    rest = data_all[10:]
    random.seed(seed)
    if len(rest) > samples:
        rest = random.sample(rest, samples)
    # remove few-shot demo indices (ids 1,3,5,7,9)
    demo_ids = {1, 3, 5, 7, 9}
    rest = [d for d in rest if d['id'] not in demo_ids]

    # align data with logits by id
    logits_by_id = {str(r['id']): r for r in logits_all}
    paired = [(d, logits_by_id[str(d['id'])]) for d in rest if str(d['id']) in logits_by_id]
    if not paired:
        return None, None, None, None

    random.seed(seed)
    random.shuffle(paired)
    n_cal = len(paired) // 2
    cal_pairs  = paired[:n_cal]
    test_pairs = paired[n_cal:]

    cal_data,  cal_logits  = zip(*cal_pairs)
    test_data, test_logits = zip(*test_pairs)
    return list(cal_data), list(cal_logits), list(test_data), list(test_logits)


print("=" * 60)
print("MONDRIAN CP  (B=3 entropy bins, LAC + APS, α=0.1)")
print("=" * 60)

mondrian_all = {}  # {short: {ds: {"lac": result, "aps": result}}}

for hf_id, short in MODELS:
    mondrian_all[short] = {}
    print(f'\n{"─"*50}')
    print(f'Model: {short}')
    print(f'{"─"*50}')

    for ds in DATASETS:
        cal_data, cal_logits, test_data, test_logits = load_split(
            short, ds, OUT_DIR, SAMPLES)
        if cal_data is None:
            print(f'  {ds}: no data, skipping')
            continue

        print(f'\n  Dataset: {ds}  (n_cal={len(cal_data)}, n_test={len(test_data)})')

        results = {}
        for score in ('lac', 'aps'):
            print(f'\n  [{score.upper()}]')
            results[score] = evaluate_mondrian(
                cal_data, cal_logits, test_data, test_logits,
                score=score, B=3, alpha=ALPHA,
            )
        mondrian_all[short][ds] = results

print('\nDone.')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# ── Comparison table: Global LAC/APS vs Mondrian LAC/APS ─────────────────────
key = f'{PROMPT}_{ICL}'
col = 10

print("GLOBAL vs MONDRIAN  (CR% / SS avg LAC+APS)")
print(f"{'Model':<14} {'Dataset':<22} {'Global CR':>10} {'Global SS':>10} "
      f"{'Mond CR':>10} {'Mond SS':>10} {'Better?':>8}")
print("-" * 90)

for hf_id, short in MODELS:
    path = f'{OUT_DIR}/{short}_all_results.json'
    if not os.path.exists(path):
        continue
    global_res = json.load(open(path))

    for ds in DATASETS:
        if ds not in global_res or key not in global_res[ds].get('Acc', {}):
            continue
        g_cr = 100 * np.mean([global_res[ds]['LAC_coverage'][key],
                               global_res[ds]['APS_coverage'][key]])
        g_ss = np.mean([global_res[ds]['LAC_set_size'][key],
                        global_res[ds]['APS_set_size'][key]])

        if short not in mondrian_all or ds not in mondrian_all[short]:
            continue
        m_res = mondrian_all[short][ds]
        m_cr = np.mean([m_res['lac']['overall']['CR'], m_res['aps']['overall']['CR']])
        m_ss = np.mean([m_res['lac']['overall']['SS'], m_res['aps']['overall']['SS']])

        better = "SS↓" if m_ss < g_ss - 0.05 else ("CR↑" if m_cr > g_cr + 0.5 else "≈")
        print(f"{short:<14} {ds:<22} {g_cr:>9.1f}% {g_ss:>9.2f}  "
              f"{m_cr:>9.1f}% {m_ss:>9.2f}  {better:>8}")

# ── Per-bin CR figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

B = 3
bin_labels = [f'Bin {b}\n({"easy" if b==0 else "med" if b==1 else "hard"})' for b in range(B)]
colors = ['#4CAF50', '#FF9800', '#F44336']

for ax, (hf_id, short) in zip(axes, MODELS):
    if short not in mondrian_all:
        ax.set_title(short)
        continue

    # average per-bin CR across datasets (LAC+APS mean)
    bin_crs = [[] for _ in range(B)]
    for ds in DATASETS:
        if ds not in mondrian_all[short]:
            continue
        for score in ('lac', 'aps'):
            per_bin = mondrian_all[short][ds][score]['per_bin']
            for b, pb in enumerate(per_bin):
                if pb is not None:
                    bin_crs[b].append(pb['CR'])

    means = [np.mean(v) if v else 0 for v in bin_crs]
    bars = ax.bar(range(B), means, color=colors, alpha=0.85, width=0.6)
    ax.axhline(90, color='red', ls='--', lw=1.2, label='90% target')
    ax.set_xticks(range(B))
    ax.set_xticklabels(bin_labels, fontsize=8)
    ax.set_ylim(0, 115)
    ax.set_title(short.replace('-', '\n'), fontsize=9, fontweight='bold')
    for bar, v in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f'{v:.0f}%',
                ha='center', va='bottom', fontsize=8)

axes[0].set_ylabel('Coverage Rate (%)', fontsize=9)
axes[0].legend(fontsize=7)
fig.suptitle('Mondrian CP — per-bin Coverage Rate (avg LAC+APS across 5 tasks)',
             fontsize=10, y=1.02)
plt.tight_layout()

out_fig = f'{FIG_DIR}/fig_mondrian_bin_cr.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_fig}')


## 7. Generate figures

In [ ]:
subprocess.run([
    sys.executable, 'plot_results.py',
    '--samples',     str(SAMPLES),
    '--prompt',      PROMPT,
    '--icl',         ICL,
    '--results_dir', OUT_DIR,
    '--figures_dir', FIG_DIR,
], check=False)

## 8. Display figures

In [ ]:
from IPython.display import Image, display
import glob

for png in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(f'\n{png}')
    display(Image(png))

## 9. Summary table

In [ ]:
import json, numpy as np

key = f'{PROMPT}_{ICL}'
col = 16
print('RESULTS  (CR% / Acc% / SS)')
print(f"{'Model':<20}" + ''.join(f"{d.split('_')[0]:>{col}}" for d in DATASETS))
print('-' * (20 + col * len(DATASETS)))

for hf_id, short in MODELS:
    path = f'{OUT_DIR}/{short}_all_results.json'
    if not os.path.exists(path):
        print(f'{short:<20}  (no results)'); continue
    res = json.load(open(path))
    row = f'{short:<20}'
    for d in DATASETS:
        if d not in res or key not in res[d].get('Acc', {}):
            row += f"{'N/A':>{col}}"; continue
        acc = 100 * res[d]['Acc'][key]
        cr  = 100 * np.mean([res[d]['LAC_coverage'][key], res[d]['APS_coverage'][key]])
        ss  =       np.mean([res[d]['LAC_set_size'][key],  res[d]['APS_set_size'][key]])
        row += f"{cr:.0f}/{acc:.0f}/{ss:.1f}".rjust(col)
    print(row)

print(f'\nn={SAMPLES} → n_cal≈52 → α={ALPHA}')

## 10. Download results

In [ ]:
import shutil

shutil.make_archive('kaggle_results', 'zip', '.', OUT_DIR)
shutil.make_archive('figures_kaggle', 'zip', '.', FIG_DIR)
print('Outputs zipped → kaggle_results.zip')
print('Figures  zipped → figures_kaggle.zip')
print('Download from: Notebook → Output tab → <filename>.zip')